In [93]:
import onnxruntime as ort
import numpy as np
from keras_image_helper import create_preprocessor

In [94]:
onnx_model_path = 'hair_classifier_v1.onnx'
session = ort.InferenceSession(onnx_model_path, providers = ['CPUExecutionProvider'])

In [95]:
inputs = session.get_inputs()
outputs = session.get_outputs()

input_name = inputs[0].name
output_name = outputs[0].name


## Q1 - Name of output node = ?
* *The name of the node is **output***

In [96]:
output_name

'output'

## Q2: Target size
* Based on previous homework, what should be the target size? *It should be **200x200***

In [97]:
from io import BytesIO
from urllib import request

from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

In [98]:
url = 'https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg'
img = download_image(url)
sized_img = prepare_image(img, (200, 200))

## Q3. Pre-processing
* After pre-processing, what's the value in first pixel R-channel?  **-1.073**

In [165]:
def tensor(img_array):
    return (img_array/255)

def norm(img_arr):
    #ImageNet normalization values from prior assignment
    means = [0.485, 0.456, 0.406]
    stds = [0.229, 0.224, 0.225]

    img_arr[..., 0] -= means[0]
    img_arr[..., 0] /= stds[0]
    img_arr[..., 1] -= means[1]
    img_arr[..., 1] /= stds[1]
    img_arr[..., 2] -= means[2]
    img_arr[..., 2] /= stds[2]

    return img_arr

def preprocess(img_array):
    img_array = np.array(img_array).astype('float32')  #make usable by Onnx
    img_array = tensor(img_array)
    print(f'Tensor shape: {img_array.shape}')
    img_array = norm(img_array)
    print(f'Normalized Tensor shape: {img_array.shape}')
    return img_array


In [166]:
processed_img = preprocess(sized_img)
processed_img = np.transpose(processed_img, (2, 0, 1))  #reshape to meet model requirements
test = np.expand_dims(processed_img, axis=0)

Tensor shape: (200, 200, 3)
Normalized Tensor shape: (200, 200, 3)


In [167]:
test.shape

(1, 3, 200, 200)

## Q4 Apply model
* What is the returned value?  *Answer is **.09***

In [174]:
#Above was generated manually, this utilizes the helper library with the same custom preprocessing function.  Same answer  (finally!)
preprocessor = create_preprocessor(preprocess, target_size=(200,200))
X = preprocessor.from_url(url)

Tensor shape: (1, 200, 200, 3)
Normalized Tensor shape: (1, 200, 200, 3)


In [169]:
X = np.transpose(X, (0, 3, 1, 2))
X.shape


(1, 3, 200, 200)

In [172]:
preds = session.run([output_name], {input_name: X})

In [173]:
preds

[array([[0.09156641]], dtype=float32)]

## Q5 What's the size of the base image?
* Size is **921MB** but closest answer is 1208.

## Q6. What is the model output?
* Output is **-0.10**

In [16]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

dict(zip(classes, float_predictions))

{'dress': -2.3684332370758057,
 'hat': -4.414736270904541,
 'longsleeve': -2.3090553283691406,
 'outwear': -1.7397485971450806,
 'pants': 8.660110473632812,
 'shirt': -3.1949868202209473,
 'shoes': -5.595435619354248,
 'shorts': 2.7487967014312744,
 'skirt': -2.9956488609313965,
 't-shirt': -4.363430500030518}